In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [4]:
path_processed = "../data/processed/"
X_train = pd.read_csv(f"{path_processed}X_train.csv")
y_train = pd.read_csv(f"{path_processed}y_train.csv").squeeze("columns")
X_test = pd.read_csv(f"{path_processed}X_test.csv")
y_test = pd.read_csv(f"{path_processed}y_test.csv").squeeze("columns")

In [5]:
RFC_opt_columns = ['CLTV',
 'Tenure Months',
 'Total Extra Services',
 'Payment Method',
 'Multiple Lines',
 'Contract',
 'Streaming TV',
 'Phone Service',
 'Paperless Billing',
 'Streaming Movies',
 'Senior Citizen',
 'Tech Support',
 'Online Security']

X_train_13 = X_train[RFC_opt_columns]
X_test_13 = X_test[RFC_opt_columns]

In [6]:
xgb_model = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
lgb_model = LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

experiments = {
    "XGBoost (13 Col)": (xgb_model, X_train_13),
    "XGBoost (18 Col)": (xgb_model, X_train),
    "LightGBM (13 Col)": (lgb_model, X_train_13),
    "LightGBM (18 Col)": (lgb_model, X_train),
}

for name, (model, dataset) in experiments.items():
    scores = cross_val_score(model, dataset, y_train, cv=5, scoring='precision', n_jobs=-1)
    avg_precision = np.mean(scores) * 100
    print(f"{name}: {avg_precision:.2f}%")

XGBoost (13 Col): 71.30%
XGBoost (18 Col): 69.74%
LightGBM (13 Col): 71.67%
LightGBM (18 Col): 71.79%


In [ ]:
param_grid_lgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, -1],
    'num_leaves': [15, 31, 63],
    'subsample': [0.8, 1.0]
}

grid_search_lgb_ = GridSearchCV(
    estimator=LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    param_grid=param_grid_lgb,
    scoring='precision',
    cv=5,
    n_jobs=-1,
    verbose=3
)

grid_search_lgb_.fit(X_train_13, y_train)

In [ ]:
print(f"\nBest Precision: {grid_search_lgb_.best_score_ * 100:.2f}%")
print(f"Best Parameters: {grid_search_lgb_.best_params_}")

In [8]:
best_lgb = LGBMClassifier(
    learning_rate=0.01,
    max_depth=3,
    n_estimators=100,
    num_leaves=15,
    subsample=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
best_lgb.fit(X_train_13, y_train)

predictions_lgb = best_lgb.predict(X_test_13)
print(f"{accuracy_score(y_test, predictions_lgb) * 100:.2f}%\n")
print(classification_report(y_test, predictions_lgb))

75.46%

              precision    recall  f1-score   support

           0       0.75      0.93      0.83       675
           1       0.77      0.42      0.54       360

    accuracy                           0.75      1035
   macro avg       0.76      0.68      0.69      1035
weighted avg       0.76      0.75      0.73      1035

